In [2]:
import os
import sys
import math
from pathlib import Path

import zarr
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

sys.path.append(str(Path('transfomer').resolve()))
from liquid_net import PushTLiquidNet

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

device: mps


In [3]:
# Config
DATASET_PATH = Path('transfomer/pusht_cchi_v7_replay.zarr')
OBS_HORIZON = 2
ACTION_HORIZON = 8
STATE_DIM = 5
ACTION_DIM = 2

CKPT_CANDIDATES = [
    Path('transfomer/checkpoints/pusht_liquid_best_trainloop.pt'),
    Path('transfomer/checkpoints/pusht_liquid.pt'),
    Path('checkpoints/liquid_jepa_pusht_fair_halfparam_deterministic_clip_best.pt'),
]

assert DATASET_PATH.exists(), f'Missing dataset: {DATASET_PATH}'
ckpt_path = next((p for p in CKPT_CANDIDATES if p.exists()), None)
assert ckpt_path is not None, f'No liquid checkpoint found in: {CKPT_CANDIDATES}'
print('dataset:', DATASET_PATH)
print('checkpoint:', ckpt_path)

dataset: transfomer/pusht_cchi_v7_replay.zarr
checkpoint: transfomer/checkpoints/pusht_liquid_best_trainloop.pt


In [4]:
# Load dataset
root = zarr.open(str(DATASET_PATH), mode='r')
states = root['data']['state'][:].astype(np.float32)   # (N,5)
actions = root['data']['action'][:].astype(np.float32) # (N,2)
episode_ends = root['meta']['episode_ends'][:]

def get_stats(x):
    return {'min': x.min(axis=0), 'max': x.max(axis=0)}

def normalize_data(x, stats):
    return (x - stats['min']) / (stats['max'] - stats['min'] + 1e-8) * 2 - 1

def unnormalize_data(x, stats):
    return (x + 1) / 2 * (stats['max'] - stats['min'] + 1e-8) + stats['min']

state_stats = get_stats(states)
action_stats = get_stats(actions)

states_n = normalize_data(states, state_stats)
actions_n = normalize_data(actions, action_stats)

# Build valid window starts (all demos)
starts = []
s = 0
for e in episode_ends:
    # need obs_horizon + action_horizon steps
    for i in range(s + OBS_HORIZON - 1, e - ACTION_HORIZON):
        starts.append(i)
    s = e

starts = np.array(starts, dtype=np.int64)
print('states:', states.shape, 'actions:', actions.shape)
print('windows:', len(starts))

states: (25650, 5) actions: (25650, 2)
windows: 23796


In [5]:
# Load trained liquid model – full-size notebook-04 architecture
# d_model=512, hidden_size=960, num_layers=5, num_mixtures=5 (matches pusht_liquid_best_trainloop.pt)
liquid = PushTLiquidNet(
    obs_horizon=OBS_HORIZON,
    action_horizon=ACTION_HORIZON,
    d_model=512,
    hidden_size=960,
    num_layers=5,
    dropout=0.1,
    num_mixtures=5,
).to(device)

obj = torch.load(ckpt_path, map_location=device, weights_only=False)

# Handle notebook-04 style checkpoint: fp16 weights stored under 'model_state_dict'
if isinstance(obj, dict) and 'model_state_dict' in obj:
    raw = obj['model_state_dict']
    state = {k: (v.float() if torch.is_floating_point(v) else v) for k, v in raw.items()}
    miss, unex = liquid.load_state_dict(state, strict=True)
    print(f'Loaded model_state_dict: {len(state)} tensors | missing={len(miss)} | unexpected={len(unex)}')
elif isinstance(obj, dict) and 'state_dict' in obj:
    miss, unex = liquid.load_state_dict(obj['state_dict'], strict=False)
    print(f'Loaded state_dict | missing={len(miss)} | unexpected={len(unex)}')
elif isinstance(obj, dict):
    miss, unex = liquid.load_state_dict(obj, strict=False)
    print(f'Loaded raw dict | missing={len(miss)} | unexpected={len(unex)}')
else:
    raise RuntimeError('Unsupported checkpoint format')

liquid.eval()
for p in liquid.parameters():
    p.requires_grad = False
print(f'Liquid params: {sum(p.numel() for p in liquid.parameters()):,}')

@torch.no_grad()
def liquid_chunk_from_obs_deque(obs_deque, stats):
    """Run liquid inference using notebook-04 inference path and stats."""
    obs_seq = np.stack(obs_deque).astype(np.float32)          # (2, 5)
    nobs_np = normalize_data(obs_seq, stats['obs']).astype(np.float32)
    nobs    = torch.from_numpy(nobs_np).to(device)
    tool_past = nobs[:, :2].unsqueeze(0)                      # (1, 2, 2)
    t_pos     = torch.cat([
        nobs[:, 2:-1],
        torch.sin(nobs[:, -1:]),
        torch.cos(nobs[:, -1:]),
    ], dim=-1).unsqueeze(0)                                   # (1, 2, 4)
    act_n = liquid(tool_past, t_pos).cpu().numpy()[0]         # (8, 2) normalized
    act   = unnormalize_data(act_n, stats['action'])          # (8, 2) pixel-space
    return act_n, act

# quick sanity check
i0    = starts[0]
hist0 = states_n[i0-OBS_HORIZON+1:i0+1]
_deq  = [hist0[i] for i in range(OBS_HORIZON)]
_n, _a = liquid_chunk_from_obs_deque(_deq, {'obs': state_stats, 'action': action_stats})
print('pred chunk shape:', tuple(_n.shape), '| pixel range:', round(float(_a.min()), 1), '–', round(float(_a.max()), 1))


Loaded model_state_dict: 59 tensors | missing=0 | unexpected=0
Liquid params: 32,103,193
pred chunk shape: (8, 2) | pixel range: -39.9 – 78.0


In [6]:
# Build RLT training pairs from ALL demonstrations (using correctly loaded liquid model)
all_token_in = []
all_base = []
all_target = []

with torch.no_grad():
    for i in starts:
        hist_n = states_n[i-OBS_HORIZON+1:i+1].astype(np.float32)   # (2,5), normalized

        # Liquid baseline chunk in normalized action space (same path as inference)
        nobs = torch.from_numpy(hist_n).to(device=device, dtype=torch.float32)
        tool_past = nobs[:, :2].unsqueeze(0)
        t_pos = torch.cat([
            nobs[:, 2:-1],
            torch.sin(nobs[:, -1:]),
            torch.cos(nobs[:, -1:]),
        ], dim=-1).unsqueeze(0)
        base = liquid(tool_past, t_pos).squeeze(0).cpu().numpy().astype(np.float32)  # (8,2), normalized

        target = actions_n[i+1:i+1+ACTION_HORIZON].astype(np.float32)                 # (8,2), normalized

        # Token input: flattened obs history + base chunk
        token_in = np.concatenate([hist_n.reshape(-1), base.reshape(-1)], axis=0)

        all_token_in.append(token_in)
        all_base.append(base.reshape(-1))
        all_target.append(target.reshape(-1))

X = torch.from_numpy(np.stack(all_token_in, axis=0))
B = torch.from_numpy(np.stack(all_base, axis=0))
Y = torch.from_numpy(np.stack(all_target, axis=0))

n = X.shape[0]
perm = torch.randperm(n)
n_train = int(0.9 * n)
tr, va = perm[:n_train], perm[n_train:]

train_ds = TensorDataset(X[tr], B[tr], Y[tr])
val_ds = TensorDataset(X[va], B[va], Y[va])

print('pairs total:', n, '| train:', len(train_ds), '| val:', len(val_ds))

pairs total: 23796 | train: 21416 | val: 2380


In [7]:
# RLT residual actor (Push-T adaptation)
class PushTRLTResidualActor(nn.Module):
    def __init__(self, token_in_dim, token_dim=64, hidden=256, out_dim=ACTION_HORIZON * ACTION_DIM):
        super().__init__()
        self.token = nn.Sequential(
            nn.Linear(token_in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, token_dim), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(token_dim + out_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x_token_in, base_flat):
        t = self.token(x_token_in)
        resid = torch.tanh(self.head(torch.cat([t, base_flat], dim=-1)))
        action = torch.clamp(base_flat + 0.3 * resid, -1.0, 1.0)
        return action

model = PushTRLTResidualActor(token_in_dim=X.shape[1]).to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=1024, shuffle=False)

for epoch in range(1, 21):
    model.train()
    tr_losses = []
    for xb, bb, yb in train_loader:
        xb, bb, yb = xb.to(device), bb.to(device), yb.to(device)
        pred = model(xb, bb)
        loss = F.mse_loss(pred, yb) + 0.01 * F.mse_loss(pred, bb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tr_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        va_losses = []
        for xb, bb, yb in val_loader:
            xb, bb, yb = xb.to(device), bb.to(device), yb.to(device)
            pred = model(xb, bb)
            va_losses.append((F.mse_loss(pred, yb) + 0.01 * F.mse_loss(pred, bb)).item())

    print(f'epoch {epoch:02d} | train={np.mean(tr_losses):.6f} | val={np.mean(va_losses):.6f}')

epoch 01 | train=0.008846 | val=0.007618
epoch 02 | train=0.006850 | val=0.006403
epoch 03 | train=0.006412 | val=0.006240
epoch 04 | train=0.006279 | val=0.006133
epoch 05 | train=0.006219 | val=0.006038
epoch 06 | train=0.006174 | val=0.006025
epoch 07 | train=0.006051 | val=0.005989
epoch 08 | train=0.006015 | val=0.005902
epoch 09 | train=0.005979 | val=0.005892
epoch 10 | train=0.005983 | val=0.005865
epoch 11 | train=0.005915 | val=0.005824
epoch 12 | train=0.005915 | val=0.005787
epoch 13 | train=0.005851 | val=0.005773
epoch 14 | train=0.005873 | val=0.005746
epoch 15 | train=0.005811 | val=0.005715
epoch 16 | train=0.005795 | val=0.005707
epoch 17 | train=0.005817 | val=0.005693
epoch 18 | train=0.005774 | val=0.005668
epoch 19 | train=0.005766 | val=0.005663
epoch 20 | train=0.005722 | val=0.005644


In [8]:
# Compare liquid baseline vs liquid+RLT (open-loop action MSE on validation windows)
model.eval()
with torch.no_grad():
    xb, bb, yb = X[va].to(device), B[va].to(device), Y[va].to(device)
    pred_rlt = model(xb, bb)
    mse_base = F.mse_loss(bb, yb).item()
    mse_rlt = F.mse_loss(pred_rlt, yb).item()

print('Validation MSE (normalized action chunks)')
print('  liquid base :', round(mse_base, 6))
print('  liquid + RLT:', round(mse_rlt, 6))

# Save artifacts
Path('checkpoints').mkdir(exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'token_in_dim': int(X.shape[1]),
    'obs_horizon': OBS_HORIZON,
    'action_horizon': ACTION_HORIZON,
    'action_dim': ACTION_DIM,
    'mse_base': mse_base,
    'mse_rlt': mse_rlt,
}, 'checkpoints/rlt_liquid_pusht_residual.pt')

print('saved -> checkpoints/rlt_liquid_pusht_residual.pt')

Validation MSE (normalized action chunks)
  liquid base : 0.00958
  liquid + RLT: 0.005703
saved -> checkpoints/rlt_liquid_pusht_residual.pt


In [9]:
# =============================================================
# Closed-loop evaluation – EXACT notebook-04 protocol
# =============================================================
import collections, json
from pathlib import Path
import numpy as np
import torch
from tqdm.auto import tqdm

# ── Pull dataset utils + PushTEnv verbatim from notebook 04 ──
_nb_src = json.loads(Path('04_pusht_closedloop.ipynb').read_text())
for _c in _nb_src['cells']:
    _s = ''.join(_c.get('source', []))
    if 'def create_sample_indices' in _s:
        exec(_s, globals())        # defines PushTStateDataset, normalize_data, unnormalize_data, …
    if 'class PushTEnv' in _s:
        exec(_s, globals())        # defines PushTEnv (pymunk physics sim)

# ── Build PushTStateDataset with notebook-04 parameters ──────
_pred_horizon   = 16
_obs_horizon    = 2
_action_horizon = 8

dataset04 = PushTStateDataset(
    dataset_path=str(DATASET_PATH),
    pred_horizon=_pred_horizon,
    obs_horizon=_obs_horizon,
    action_horizon=_action_horizon,
)
stats04 = dataset04.stats   # {'obs': {min, max}, 'action': {min, max}}

# Replicate notebook-04 80/20 split (same seed for reproducibility)
_n_train = int(0.8 * len(dataset04))
_n_val   = len(dataset04) - _n_train
torch.manual_seed(42)
_, val_dataset04 = torch.utils.data.random_split(dataset04, [_n_train, _n_val])
print(f"PushTStateDataset  total={len(dataset04):,}  val={len(val_dataset04):,}")

# Sanity: env + liquid model
_env_t = PushTEnv(); _env_t.seed(0); _o0, _ = _env_t.reset()
print(f"PushTEnv OK | obs shape: {_o0.shape}")
_samp0 = val_dataset04[0]
_oh    = unnormalize_data(_samp0['obs'], stats04['obs'])
_n0, _a0 = liquid_chunk_from_obs_deque(list(_oh), stats04)
print(f"liquid sanity: act_n range [{_n0.min():.2f}, {_n0.max():.2f}]  pixel range [{_a0.min():.0f}, {_a0.max():.0f}]")

# ── Inference helpers (notebook-04 inference path, no action clip) ──
@torch.no_grad()
def _infer_liquid(obs_deque):
    """Returns (act_n, act): normalized (8,2) and pixel-space (8,2)."""
    return liquid_chunk_from_obs_deque(obs_deque, stats04)

@torch.no_grad()
def _infer_rlt(obs_deque):
    """Liquid base + RLT residual correction."""
    act_n, _ = _infer_liquid(obs_deque)
    nobs_np   = normalize_data(np.stack(obs_deque).astype(np.float32), stats04['obs'])
    tok_in    = np.concatenate([nobs_np.reshape(-1), act_n.reshape(-1)]).astype(np.float32)
    t_tok = torch.from_numpy(tok_in).unsqueeze(0).to(device)
    b_tok = torch.from_numpy(act_n.reshape(-1)).unsqueeze(0).to(device)
    rlt_n = model(t_tok, b_tok).squeeze(0).cpu().numpy().reshape(_action_horizon, 2)
    rlt   = unnormalize_data(rlt_n, stats04['action'])   # pixel-space
    return rlt

# ── Evaluation loop – verbatim notebook-04 logic ─────────────
def run_eval_nb04(policy='liquid', num_samples=None):
    liquid.eval(); model.eval()
    n = num_samples or len(val_dataset04)
    successes, max_rewards = 0, []

    with tqdm(total=n, desc=f"rollout [{policy}]") as pbar:
        for idx in range(n):
            sample   = val_dataset04[idx]
            obs_hist = unnormalize_data(sample['obs'], stats04['obs'])  # (2,5) env coords

            env = PushTEnv(reset_to_state=obs_hist[-1])
            env.seed(1000 + idx)
            env.reset()

            obs_deque = collections.deque([o.copy() for o in obs_hist], maxlen=_obs_horizon)
            done, step_idx, rewards = False, 0, []

            while not done and step_idx < 200:
                if policy == 'liquid':
                    _, chunk = _infer_liquid(obs_deque)
                else:
                    chunk = _infer_rlt(obs_deque)

                for a in chunk:           # ← raw pixel coords, NO clip (critical!)
                    obs, reward, done, _, _ = env.step(a)
                    obs_deque.append(obs)
                    rewards.append(float(reward))
                    step_idx += 1
                    if done or step_idx >= 200:
                        done = True
                        break

            max_r = max(rewards) if rewards else 0.0
            max_rewards.append(max_r)
            if max_r >= 0.95:
                successes += 1
            pbar.set_postfix(sr=f"{100*successes/(idx+1):.1f}%")
            pbar.update(1)

    return {
        'policy': policy,
        'n': n,
        'success_rate': successes / max(1, n),
        'avg_max_reward': float(np.mean(max_rewards)),
    }

N_EVAL = len(val_dataset04)
liquid_res = run_eval_nb04('liquid', N_EVAL)
rlt_res    = run_eval_nb04('rlt',    N_EVAL)

print(f"\nLiquid baseline : {liquid_res['success_rate']*100:.1f}%  avg_max_r={liquid_res['avg_max_reward']:.4f}")
print(f"Liquid + RLT    : {rlt_res['success_rate']*100:.1f}%  avg_max_r={rlt_res['avg_max_reward']:.4f}")

Path('artifacts').mkdir(exist_ok=True)
out = Path('artifacts/rlt_liquid_pusht_closedloop_nb04.json')
out.write_text(json.dumps({'liquid': liquid_res, 'rlt': rlt_res}, indent=2))
print(f"saved → {out}")


Dataset utilities defined.
PushTEnv sanity check OK | obs shape: (5,) | reward: 0.1878
PushTStateDataset  total=24,208  val=4,842
PushTEnv OK | obs shape: (5,)
liquid sanity: act_n range [-0.07, 0.45]  pixel range [243, 377]


rollout [liquid]:   0%|          | 0/4842 [00:00<?, ?it/s]

rollout [rlt]:   0%|          | 0/4842 [00:00<?, ?it/s]


Liquid baseline : 87.0%  avg_max_r=0.9659
Liquid + RLT    : 66.1%  avg_max_r=0.9091
saved → artifacts/rlt_liquid_pusht_closedloop_nb04.json


In [ ]:
# =============================================================
# AWAC: Advantage-Weighted Actor-Critic
# Conservative offline RL that only moves the residual actor
# toward actions empirically BETTER than the liquid baseline.
#
# Key idea:
#   For each state s, run liquid + N random perturbations.
#   Compute advantage A_k = return(perturb_k) - return(liquid).
#   Train actor with weighted-BC: loss = exp(A_k / beta) * MSE(actor, perturb_k)
#   -> high-advantage (better-than-liquid) actions get high weight
#   -> low/negative advantage actions get downweighted / ignored
#   -> actor NEVER extrapolates beyond observed rollouts (conservative)
# =============================================================
assert 'val_dataset04' in globals(), 'Run Cell 8 first.'

# ---------- hypers ----------
N_STATES     = 400      # env states to sample from val split
N_PERTURB    = 5        # perturbations per state (+ liquid baseline = N_PERTURB+1 total)
PERTURB_SCALE = 0.12    # std of Gaussian perturbation in normalized action space
BETA          = 0.5     # AWAC temperature; higher = more conservative
LAMBDA_BC     = 0.40    # BC anchor weight (keeps actor near liquid baseline)
AWAC_EPOCHS   = 25      # fine-tuning epochs after data collection
LR_AWAC       = 8e-5    # conservative LR

# Reload BC-warm-started model (before critic fine-tuning corrupted it)
model_awac = PushTRLTResidualActor(token_in_dim=X.shape[1]).to(device)
_bc_ckpt = torch.load('checkpoints/rlt_liquid_pusht_residual.pt', map_location=device, weights_only=False)
model_awac.load_state_dict(_bc_ckpt['model_state_dict'])
model_awac.eval()
print('AWAC actor warm-started from BC checkpoint.')

# ---------- Step 1: collect rollout buffer ----------
buf_obs_n = []    # normalized obs history, shape (OBS_HORIZON*STATE_DIM,)
buf_base_n = []   # liquid action (normalized flat), shape (ACTION_HORIZON*ACTION_DIM,)
buf_cand_n = []   # candidate action (normalized flat)
buf_adv = []      # advantage = return(cand) - return(liquid)

chosen_idx = np.random.choice(len(val_dataset04), size=min(N_STATES, len(val_dataset04)), replace=False)

for ii, idx in enumerate(tqdm(chosen_idx, desc='AWAC rollouts')):
    sample = val_dataset04[int(idx)]
    obs_hist = unnormalize_data(sample['obs'], stats04['obs']).astype(np.float32)  # (2,5) pixel
    obs_n    = normalize_data(obs_hist, stats04['obs']).astype(np.float32)          # (2,5) normalized

    obs_deque = collections.deque([o.copy() for o in obs_hist], maxlen=OBS_HORIZON)
    act_n_liq, act_px_liq = _infer_liquid(obs_deque)

    # baseline return: run liquid action for first chunk, then liquid to completion
    r_liq = _rollout_return_with_first_chunk(obs_hist, act_px_liq, max_steps=200)

    # candidate perturbations of the liquid action
    candidates_n = []
    for _ in range(N_PERTURB):
        noise = np.random.normal(0, PERTURB_SCALE, act_n_liq.shape).astype(np.float32)
        cand = np.clip(act_n_liq + noise, -1.0, 1.0)
        candidates_n.append(cand)

    obs_flat = obs_n.reshape(-1)
    base_flat = act_n_liq.reshape(-1)

    for cand_n in candidates_n:
        cand_px = unnormalize_data(cand_n, stats04['action']).astype(np.float32)
        r_cand  = _rollout_return_with_first_chunk(obs_hist, cand_px, max_steps=200)
        adv     = float(r_cand) - float(r_liq)

        buf_obs_n.append(obs_flat)
        buf_base_n.append(base_flat)
        buf_cand_n.append(cand_n.reshape(-1))
        buf_adv.append(adv)

buf_obs_n  = torch.from_numpy(np.stack(buf_obs_n).astype(np.float32)).to(device)
buf_base_n = torch.from_numpy(np.stack(buf_base_n).astype(np.float32)).to(device)
buf_cand_n = torch.from_numpy(np.stack(buf_cand_n).astype(np.float32)).to(device)
buf_adv    = torch.tensor(buf_adv, dtype=torch.float32, device=device)

# AWAC weights: exp(A/beta), per sample (no per-state normalization needed here;
# negatives automatically get weight < 1, positives get weight > 1)
awac_w = torch.exp(buf_adv.clamp(-5.0, 5.0) / BETA)
awac_w = awac_w / awac_w.mean()   # normalize for stable LR

n_pos = (buf_adv > 0).sum().item()
print(f'\nBuffer: {len(buf_adv)} transitions | {n_pos} better-than-liquid ({100*n_pos/len(buf_adv):.0f}%)')
print(f'Advantage: mean={buf_adv.mean().item():.4f}  std={buf_adv.std().item():.4f}')
print(f'AWAC weight: min={awac_w.min().item():.3f}  max={awac_w.max().item():.3f}')

# Build token input for model: [obs_n_flat, base_n_flat]
buf_token_in = torch.cat([buf_obs_n, buf_base_n], dim=-1)   # (N, 10 + 16)

# ---------- Step 2: offline AWAC pairs (BC data) for regularization ----------
X_dev = X.to(device)
B_dev = B.to(device)
Y_dev = Y.to(device)   # demo actions (all close to liquid)

# ---------- Step 3: fine-tune with AWAC objective ----------
opt_awac = torch.optim.Adam(model_awac.parameters(), lr=LR_AWAC)

for epoch in range(1, AWAC_EPOCHS + 1):
    model_awac.train()
    perm = torch.randperm(len(buf_adv), device=device)
    losses_awac, losses_bc = [], []

    for s in range(0, len(perm), 256):
        idx = perm[s:s+256]
        tok_b = buf_token_in[idx]
        bas_b = buf_base_n[idx]
        cand_b = buf_cand_n[idx]
        w_b    = awac_w[idx]

        pred = model_awac(tok_b, bas_b)

        # AWAC loss: advantage-weighted MSE to better-than-liquid candidate actions
        per_sample_mse = ((pred - cand_b) ** 2).mean(dim=-1)
        loss_awac = (w_b * per_sample_mse).mean()

        # BC regularization: on demo pairs, stay near demo actions
        bi = torch.randint(0, X_dev.shape[0], (min(256, X_dev.shape[0]),), device=device)
        pred_bc = model_awac(X_dev[bi], B_dev[bi])
        loss_bc = F.mse_loss(pred_bc, Y_dev[bi]) + 0.05 * F.mse_loss(pred_bc, B_dev[bi])

        loss = loss_awac + LAMBDA_BC * loss_bc

        opt_awac.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_awac.parameters(), 1.0)
        opt_awac.step()

        losses_awac.append(loss_awac.item())
        losses_bc.append(loss_bc.item())

    if epoch % 5 == 0 or epoch == 1:
        print(f'AWAC epoch {epoch:02d} | awac={np.mean(losses_awac):.6f} | bc={np.mean(losses_bc):.6f}')

model_awac.eval()

# ---------- Step 4: swap model -> model_awac for eval ----------
_old_model = model
model = model_awac

quick_n = 1000
liq_res = run_eval_nb04('liquid', num_samples=quick_n)
rlt_res = run_eval_nb04('rlt',    num_samples=quick_n)

print(f"\nLiquid baseline : {liq_res['success_rate']*100:.1f}%  avg_max_r={liq_res['avg_max_reward']:.4f}")
print(f"Liquid + AWAC   : {rlt_res['success_rate']*100:.1f}%  avg_max_r={rlt_res['avg_max_reward']:.4f}")

# Save
torch.save({'model_state_dict': model_awac.state_dict()},
           'checkpoints/rlt_liquid_pusht_awac.pt')
Path('artifacts').mkdir(exist_ok=True)
Path('artifacts/rlt_liquid_pusht_closedloop_awac.json').write_text(
    json.dumps({'liquid': liq_res, 'rlt': rlt_res}, indent=2))
print('saved → checkpoints/rlt_liquid_pusht_awac.pt')
print('saved → artifacts/rlt_liquid_pusht_closedloop_awac.json')

Build critic targets:   0%|          | 0/500 [00:00<?, ?it/s]

critic dataset: torch.Size([2500, 10]) torch.Size([2500, 16]) torch.Size([2500]) | return mean= 0.962435245513916
critic epoch 01 | mse=0.400909
critic epoch 05 | mse=0.029062
critic epoch 10 | mse=0.017541
critic epoch 15 | mse=0.014945
critic epoch 20 | mse=0.013446
critic epoch 25 | mse=0.012856
critic epoch 30 | mse=0.012177
actor ft epoch 01 | loss=-1.000617 | q=1.0026
actor ft epoch 02 | loss=-1.086440 | q=1.0919
actor ft epoch 03 | loss=-1.171808 | q=1.1835
actor ft epoch 04 | loss=-1.221263 | q=1.2386
actor ft epoch 05 | loss=-1.232919 | q=1.2520
actor ft epoch 06 | loss=-1.235924 | q=1.2553
actor ft epoch 07 | loss=-1.237105 | q=1.2566
actor ft epoch 08 | loss=-1.237991 | q=1.2576
actor ft epoch 09 | loss=-1.238464 | q=1.2582
actor ft epoch 10 | loss=-1.238727 | q=1.2585
actor ft epoch 11 | loss=-1.239200 | q=1.2590
actor ft epoch 12 | loss=-1.239355 | q=1.2592
actor ft epoch 13 | loss=-1.239657 | q=1.2595
actor ft epoch 14 | loss=-1.239427 | q=1.2593
actor ft epoch 15 | loss=

rollout [liquid]:   0%|          | 0/1000 [00:00<?, ?it/s]

rollout [rlt]:   0%|          | 0/1000 [00:00<?, ?it/s]


[critic-guided quick eval]
Liquid baseline : 86.3%  avg_max_r=0.9612
Liquid + RLT    : 0.8%  avg_max_r=0.3465
saved -> artifacts/rlt_liquid_pusht_closedloop_nb04_criticquick.json
